In [17]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_groq import ChatGroq
from dotenv import load_dotenv

In [18]:
load_dotenv()

True

In [19]:
llm = ChatGroq(
    model = "openai/gpt-oss-20b",
    temperature = 0.4
)

In [39]:
class QuestionWork(TypedDict):
    question : str
    answer : str
    question_category : str

In [40]:
def analyzed_question(state: QuestionWork) -> QuestionWork:
    question = state["question"]

    prompt = f"""
    Analyze the following question and identify its category.
    Return only the category name.

    Question: {question}
    """

    result = llm.invoke(prompt)

    return {
        **state,
        "question_category": result.content
    }

In [41]:
def response_func(state: QuestionWork) -> QuestionWork:
    question = state["question"]

    prompt = f"Answer the following question: {question}"

    result = llm.invoke(prompt)

    return {
        **state,
        "answer": result.content
    }

In [44]:
graph = StateGraph(QuestionWork)

graph.add_node("response_func", response_func)
graph.add_node("analyzed_question", analyzed_question)

graph.add_edge(START, "analyzed_question")
graph.add_edge("analyzed_question", "response_func")
graph.add_edge("response_func", END)

workflow = graph.compile()


In [49]:
inital_state = {"question" :"what is AI?"}
final_state = workflow.invoke(inital_state)
print(final_state)

{'question': 'what is AI?', 'answer': '**Artificial Intelligence (AI)** is a branch of computer science that focuses on creating systems capable of performing tasks that normally require human intelligence. These tasks include:\n\n| Core Capability | Typical Examples |\n|-----------------|------------------|\n| **Perception** | Image and speech recognition, natural language processing |\n| **Reasoning** | Logical inference, planning, problem solving |\n| **Learning** | Machine learning, deep learning, reinforcement learning |\n| **Interaction** | Dialogue systems, chatbots, autonomous agents |\n| **Autonomy** | Self‑driving cars, robotic manipulation, adaptive game AI |\n\n### Key Points\n\n1. **Goal** – Build machines that can *understand*, *learn*, *plan*, and *act* in complex environments.\n2. **Approaches** –  \n   * *Symbolic AI* (rule‑based, logic, knowledge graphs)  \n   * *Statistical/Connectionist AI* (neural networks, probabilistic models)  \n   * *Hybrid systems* that combin